In [0]:
bronze_path_movies = "/Volumes/workspace/default/myspace/movies/bronze/movies/movies.parquet"
bronze_path_ratings = "/Volumes/workspace/default/myspace/movies/bronze/ratings/ratings.parquet"

movies_df = (spark.read.format("parquet")
      .option("recursiveFileLookup", "true")
      .load(bronze_path_movies)
)

ratings_df = (spark.read.format("parquet")
      .option("recursiveFileLookup", "true")
      .load(bronze_path_ratings)
)

In [0]:
movies_df.show(5)

In [0]:
ratings_df.show(5)

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import  TimestampType

In [0]:
# split genres values
movies_df  =movies_df.withColumn('splitted_genres',F.split(F.col('genres'), '\\|'))

# explode on genres
movies_df  =movies_df.withColumn('genre', F.explode('splitted_genres'))

# remove unnecessary columns
movies_df = movies_df.drop(*['genres', 'splitted_genres'])


In [0]:
movies_df.show(10)

In [0]:
# remove records with rating less than 1
# convert timestamp to datetime type
# add year, month and day columns into data

ratings_df = (ratings_df.filter(F.col('rating') >= 1)
                        .withColumn("datetime", F.from_unixtime(F.col("timestamp")).cast(TimestampType()))
                        .drop("timestamp")
                        .withColumnRenamed("datetime", "timestamp")        
                        .withColumn("year", F.year("timestamp"))
                        .withColumn("month", F.month("timestamp"))
                        .withColumn("day", F.dayofmonth("timestamp"))
)

In [0]:
ratings_df.show(5)

In [0]:
silver_path_movies = "/Volumes/workspace/default/myspace/movies/silver/movies/movies.parquet"
silver_path_ratings = "/Volumes/workspace/default/myspace/movies/silver/ratings/ratings.parquet"

movies_df.write.parquet(path=silver_path_movies,
                        mode='overwrite')

ratings_df.write.parquet(path=silver_path_ratings,
                        mode='overwrite')


=============================================== END =============================================